# Notebook 1: Introduction to CUDA

Welcome to the CUDA C++ course! In this first notebook, you will learn:

- What GPU computing is and why it matters
- How CPUs and GPUs differ architecturally
- The CUDA programming model at a high level
- How to set up your environment for CUDA development in Jupyter
- Your very first CUDA program

## 1.1 Why GPU Computing?

Modern CPUs are optimized for **sequential tasks** — they have a few powerful cores (4–16 typically) with large caches, sophisticated branch prediction, and out-of-order execution.

GPUs take the opposite approach: they have **thousands of simple cores** designed to do the same operation on many data elements simultaneously. This is called **data parallelism**.

```
CPU: 8 powerful cores           GPU: 4,000+ simple cores
┌────────────────────┐          ┌──────────────────────────┐
│ ████  ████         │          │ ▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪ │
│ ████  ████         │          │ ▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪ │
│ ████  ████         │          │ ▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪ │
│ ████  ████         │          │ ▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪ │
│                    │          │ ▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪ │
│  Large caches      │          │ ▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪ │
│  Complex control   │          │  Small caches             │
│  Low latency       │          │  Simple control            │
│                    │          │  High throughput           │
└────────────────────┘          └──────────────────────────┘
Best for: serial tasks          Best for: parallel tasks
```

**When to use a GPU:**
- Same operation applied to millions of data elements (images, vectors, matrices)
- Scientific simulations, deep learning, signal processing, graphics

**When NOT to use a GPU:**
- Heavily branching serial code
- Small data that doesn't justify transfer overhead
- Tasks requiring complex control flow

## 1.2 What is CUDA?

**CUDA** (Compute Unified Device Architecture) is NVIDIA's parallel computing platform. It lets you write C/C++ code that runs on NVIDIA GPUs.

Key terminology:

| Term | Meaning |
|------|--------|
| **Host** | The CPU and its memory (RAM) |
| **Device** | The GPU and its memory (VRAM) |
| **Kernel** | A function that runs on the GPU |
| **Thread** | One execution unit on the GPU |
| **Block** | A group of threads that can cooperate |
| **Grid** | A collection of blocks that form one kernel launch |

The basic CUDA workflow:
1. Allocate memory on the GPU
2. Copy data from CPU → GPU
3. Launch a kernel (GPU function) with many threads
4. Copy results from GPU → CPU
5. Free GPU memory

## 1.3 Environment Setup

We use the `nvcc4jupyter` extension to compile and run CUDA code directly in Jupyter cells. Let's set it up.

In [ ]:
# Install nvcc4jupyter if not already installed
!pip install nvcc4jupyter -q

In [ ]:
# Load the CUDA extension
%load_ext nvcc4jupyter

Let's first check what GPU hardware is available:

In [ ]:
!nvidia-smi

In [ ]:
!nvcc --version

## 1.4 Querying GPU Properties from CUDA

CUDA provides a runtime API to query detailed GPU information. This is useful for understanding what your hardware can do.

In [ ]:
%%cuda
#include <cstdio>

int main() {
    int deviceCount = 0;
    cudaGetDeviceCount(&deviceCount);
    printf("Number of CUDA devices: %d\n\n", deviceCount);

    for (int i = 0; i < deviceCount; i++) {
        cudaDeviceProp prop;
        cudaGetDeviceProperties(&prop, i);

        printf("Device %d: %s\n", i, prop.name);
        printf("  Compute capability:    %d.%d\n", prop.major, prop.minor);
        printf("  Total global memory:   %.1f GB\n", prop.totalGlobalMem / 1e9);
        printf("  SM count:              %d\n", prop.multiProcessorCount);
        printf("  Max threads per block: %d\n", prop.maxThreadsPerBlock);
        printf("  Max block dimensions:  (%d, %d, %d)\n",
               prop.maxThreadsDim[0], prop.maxThreadsDim[1], prop.maxThreadsDim[2]);
        printf("  Max grid dimensions:   (%d, %d, %d)\n",
               prop.maxGridSize[0], prop.maxGridSize[1], prop.maxGridSize[2]);
        printf("  Warp size:             %d\n", prop.warpSize);
        printf("  Clock rate:            %.2f GHz\n", prop.clockRate / 1e6);
        printf("  Memory clock rate:     %.2f GHz\n", prop.memoryClockRate / 1e6);
        printf("  Memory bus width:      %d bits\n", prop.memoryBusWidth);
    }
    return 0;
}

### Key numbers to remember:
- **SM (Streaming Multiprocessor)**: The basic processing unit. More SMs = more parallel work.
- **Warp size**: 32. Threads execute in groups of 32 called warps. This is fundamental.
- **Max threads per block**: Usually 1024. You'll use 128–512 in practice.
- **Compute capability**: Determines which features are available (e.g., 7.0+ for Tensor Cores).

## 1.5 Your First CUDA Program — Hello from the GPU!

Let's write the simplest possible CUDA program. A **kernel** is a function prefixed with `__global__` that runs on the GPU.

In [ ]:
%%cuda
#include <cstdio>

// This is a CUDA kernel — it runs on the GPU
__global__ void hello_cuda() {
    printf("Hello from the GPU!\n");
}

int main() {
    printf("Hello from the CPU!\n");

    // Launch the kernel with 1 block of 1 thread
    hello_cuda<<<1, 1>>>();

    // Wait for the GPU to finish
    cudaDeviceSynchronize();

    printf("Back on the CPU!\n");
    return 0;
}

### Anatomy of this program:

```cpp
__global__ void hello_cuda()    // __global__ = runs on GPU, called from CPU
hello_cuda<<<1, 1>>>();         // <<<blocks, threads_per_block>>>
cudaDeviceSynchronize();        // CPU waits for GPU to finish
```

**Important:** Kernel launches are **asynchronous**. The CPU doesn't wait for the GPU to finish unless you explicitly call `cudaDeviceSynchronize()`. Without it, the program might exit before the GPU prints anything.

## 1.6 Launching Multiple Threads

The real power of CUDA is launching **many threads** at once. Let's launch 8 threads and see them all run.

In [ ]:
%%cuda
#include <cstdio>

__global__ void hello_threads() {
    // threadIdx.x is the thread's index within its block
    printf("Hello from thread %d!\n", threadIdx.x);
}

int main() {
    // Launch 1 block of 8 threads
    hello_threads<<<1, 8>>>();
    cudaDeviceSynchronize();
    return 0;
}

**Notice:** The threads may print in any order! GPU threads execute in parallel, and there's no guaranteed ordering between them. This is a fundamental concept in parallel programming.

Now let's try multiple **blocks**:

In [ ]:
%%cuda
#include <cstdio>

__global__ void hello_blocks() {
    // blockIdx.x  = which block this thread is in
    // threadIdx.x = which thread within the block
    printf("Block %d, Thread %d\n", blockIdx.x, threadIdx.x);
}

int main() {
    // Launch 4 blocks, each with 2 threads = 8 threads total
    hello_blocks<<<4, 2>>>();
    cudaDeviceSynchronize();
    return 0;
}

## 1.7 CUDA Function Qualifiers

CUDA adds three function qualifiers:

| Qualifier | Runs on | Called from | Notes |
|-----------|---------|-------------|-------|
| `__global__` | GPU | CPU (or GPU with dynamic parallelism) | Must return `void` |
| `__device__` | GPU | GPU only | Helper functions for kernels |
| `__host__` | CPU | CPU | Regular C++ function (default) |

You can combine `__host__ __device__` to compile a function for both CPU and GPU.

In [ ]:
%%cuda
#include <cstdio>

// Runs on GPU only — can be called from kernels
__device__ int square(int x) {
    return x * x;
}

// Runs on both CPU and GPU
__host__ __device__ int cube(int x) {
    return x * x * x;
}

__global__ void compute_kernel() {
    int tid = threadIdx.x;
    printf("Thread %d: square=%d, cube=%d\n", tid, square(tid), cube(tid));
}

int main() {
    // Use cube() on CPU
    printf("CPU: cube(3) = %d\n", cube(3));

    // Use both on GPU
    compute_kernel<<<1, 5>>>();
    cudaDeviceSynchronize();
    return 0;
}

## 1.8 Key Takeaways

1. **GPUs** excel at data-parallel tasks — thousands of threads doing the same work on different data
2. **CUDA** lets you write GPU code in C/C++ using special syntax
3. **`__global__`** marks a function as a kernel (runs on GPU, called from CPU)
4. **`<<<blocks, threads>>>`** is how you launch a kernel with parallel threads
5. **`cudaDeviceSynchronize()`** waits for the GPU to finish
6. GPU threads execute in **parallel** with **no guaranteed order**
7. Every thread knows its identity via `threadIdx.x` and `blockIdx.x`

## Exercises

**Exercise 1.1:** Modify the hello kernel to launch 16 blocks of 16 threads each. Print both the block and thread index. How many total threads are launched?

**Exercise 1.2:** Write a kernel that computes the global thread ID as `blockIdx.x * blockDim.x + threadIdx.x` and prints it. Launch with 3 blocks of 4 threads. Verify the IDs go from 0 to 11.

**Exercise 1.3:** Write a `__device__` helper function that computes `n!` (factorial). Call it from a kernel that launches 8 threads, each printing `factorial(threadIdx.x)`.

In [ ]:
%%cuda
// Exercise 1.1 — Your code here
#include <cstdio>

__global__ void hello_many() {
    // TODO: Print block index and thread index
}

int main() {
    // TODO: Launch with 16 blocks of 16 threads
    cudaDeviceSynchronize();
    return 0;
}

In [ ]:
%%cuda
// Exercise 1.2 — Your code here
#include <cstdio>

__global__ void global_id() {
    // TODO: Compute and print global thread ID
}

int main() {
    // TODO: Launch with 3 blocks of 4 threads
    cudaDeviceSynchronize();
    return 0;
}

In [ ]:
%%cuda
// Exercise 1.3 — Your code here
#include <cstdio>

__device__ int factorial(int n) {
    // TODO: Implement factorial
    return 1;
}

__global__ void factorial_kernel() {
    // TODO: Print factorial of threadIdx.x
}

int main() {
    factorial_kernel<<<1, 8>>>();
    cudaDeviceSynchronize();
    return 0;
}

---
**Next:** [Notebook 2 — Kernels and Thread Hierarchy](02_Kernels_and_Thread_Hierarchy.ipynb) — Deep dive into how threads are organized and indexed.